# PartImageNet bridge demo

The composition evaluation assigns every PartImageNet supercategory a global concept `i*` and every part a local concept `j*`. This notebook shows what those concepts respond to, using the trained LG-SAE model and the matches stored in the evaluation results.

The figures follow the PartImageNet test split in feature-row order:

1. Encode every image.
2. For each supercategory, show the images where `i*` activates most.
3. For each part, show the images where the mean-pooled `j*` activates most, with that concept's spatial code overlaid.
4. Open one part image by image, then decompose one supercategory through the bridge into the local concepts with the largest weights.


## Setup


In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import yaml
from PIL import Image
from torchvision.transforms import CenterCrop, Compose, InterpolationMode, Resize, ToTensor
from tqdm import tqdm

ROOT = Path.cwd().resolve()
if not (ROOT / "lg_sae").is_dir() and (ROOT.parent / "lg_sae").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from lg_sae.partimagenet import load_features, load_ontology, patch_grid_size
from lg_sae.sae import COMPONENTS, LocalGlobalSAE, build_sae, load_components

# Stage-3 SAE, and the composition results that supply i* and j*.
SAE_CHECKPOINT = ROOT / "saved_models/clip_batchtopk_variant4/checkpoint_stage3.pt"
MODEL_CONFIG = ROOT / "saved_models/clip_batchtopk_variant4/config.json"
ONTOLOGY_EVAL_JSON = ROOT / "PartImageNet_composition_eval/clip_batchtopk_variant4/results.json"

# Flat PartImageNet test directory. Feature paths are read from the eval results.
IMAGES_DIR = Path("/gfs/shared/itaybenou/raid/datasets/PartImageNet/images/test")

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
IMAGE_SIZE = 224
TOP_K = 10
BATCH_SIZE = 32

# EXAMPLE_SUPERCATEGORY selects the local-concept rows and the bridge section.
# EXAMPLE_PART selects the one-image-per-figure section.
EXAMPLE_PART = 13
EXAMPLE_SUPERCATEGORY = 3
BRIDGE_LOCALS = 10


## Features, matches, and the SAE

Row `i` of the features is `image_ids[i]`. Displayed images use CLIP's resize and center crop without color normalization, so a 14×14 code map lines up with the picture.


In [ ]:
with open(ONTOLOGY_EVAL_JSON) as f:
    results = json.load(f)
recorded = results["config"]

patch_tokens, global_features, image_ids = load_features(
    recorded["patch_tokens"], recorded["global_features"], recorded["image_ids"])
ontology = load_ontology(recorded["ontology"])
grid = patch_grid_size(patch_tokens.shape[1])

image_paths = {}
for name in os.listdir(IMAGES_DIR):
    stem, ext = os.path.splitext(name)
    if ext.lower() in {".jpg", ".jpeg", ".png"}:
        image_paths[stem] = os.path.join(IMAGES_DIR, name)
missing = [image_id for image_id in image_ids if image_id not in image_paths]
if missing:
    raise FileNotFoundError(f"{len(missing)} image ids are missing from {IMAGES_DIR}, e.g. {missing[:3]}")

i_star = results["i_star"]
j_star = results["j_star"]
if len(i_star) != ontology.n_supercats or len(j_star) != ontology.n_parts:
    raise ValueError(
        f"i* has {len(i_star)} entries for {ontology.n_supercats} supercategories; "
        f"j* has {len(j_star)} entries for {ontology.n_parts} parts")

display_transform = Compose([
    Resize(IMAGE_SIZE, interpolation=InterpolationMode.BICUBIC),
    CenterCrop(IMAGE_SIZE),
    ToTensor(),
])


def load_image(index):
    """CLIP-cropped image tensor, (3, H, W) in [0, 1], and its id."""
    image = Image.open(image_paths[image_ids[index]]).convert("RGB")
    return display_transform(image), image_ids[index]


with open(MODEL_CONFIG) as f:
    model_cfg = yaml.safe_load(f)
model = LocalGlobalSAE(
    local_sae=build_sae(model_cfg["sae_type"], patch_tokens.shape[2], model_cfg["n_local"],
                        model_cfg["k_local"], model_cfg["local_encoder"],
                        model_cfg["threshold_momentum"], DEVICE),
    global_sae=build_sae(model_cfg["sae_type"], global_features.shape[1], model_cfg["n_global"],
                         model_cfg["k_global"], model_cfg["global_encoder"],
                         model_cfg["threshold_momentum"], DEVICE),
    pooling=model_cfg["pooling"], c_top_k=model_cfg["c_top_k"])
load_components(model, SAE_CHECKPOINT, COMPONENTS, DEVICE)
model.eval()

print(f"{len(image_ids)} images, patch grid {grid}x{grid}, device {DEVICE}")
print(f"SAE: {type(model.local_sae).__name__}, n_local={model.n_local}, n_global={model.n_global}, "
      f"pooling={model.pooling}")
print(f"composition: precision={results['precision']:.3f}  recall={results['recall']:.3f}  "
      f"F1={results['f1']:.3f}  (TP={results['tp']}, FP={results['fp']}, FN={results['fn']})")


## Codes on the test split

`z_global[i, g]` is the global code of image `i`. `z_local_pooled[i, j]` is the mean of local concept `j` over that image's patches, and it is the score used to rank images. Heatmaps use the unpooled code of the same concept.


In [ ]:
@torch.no_grad()
def encode_split(model, global_features, patch_tokens, batch_size, device):
    z_global, z_local_pooled = [], []
    for start in tqdm(range(0, global_features.shape[0], batch_size), desc="encode"):
        images = global_features[start:start + batch_size].to(device).float()
        patches = patch_tokens[start:start + batch_size].to(device).float()
        z_global.append(model.encode_global(images)[1].float().cpu())
        _, z_local = model.encode_local(patches)
        z_local_pooled.append(model.pool(z_local, patches.shape[0]).float().cpu())
    return torch.cat(z_global), torch.cat(z_local_pooled)


z_global, z_local_pooled = encode_split(model, global_features, patch_tokens, BATCH_SIZE, DEVICE)
print(f"z_global {tuple(z_global.shape)}, z_local_pooled {tuple(z_local_pooled.shape)}")


## Matched concepts

`i*` is the global concept assigned to a supercategory. `j*` is the local concept assigned to a part.


In [ ]:
print(f"{'supercategory':<16} i*")
for name, concept in zip(ontology.supercat_names, i_star):
    print(f"{name:<16} {concept}")

print()
print(f"{'part':<28} {'supercategory':<16} j*")
for part, concept in enumerate(j_star):
    supercat = ontology.supercat_names[int(ontology.part_to_supercat[part])]
    print(f"{ontology.part_names[part]:<28} {supercat:<16} {concept}")


## Figures


In [ ]:
def top_indexes(scores, k):
    return torch.topk(scores, k=min(k, scores.shape[0])).indices.tolist()


def upsample(heatmap):
    """(grid, grid) code map -> (IMAGE_SIZE, IMAGE_SIZE) array."""
    field = heatmap.detach().float().to(DEVICE)[None, None]
    return F.interpolate(field, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False)[0, 0].cpu().numpy()


@torch.no_grad()
def local_heatmaps(indexes, concept):
    patches = patch_tokens[indexes].to(DEVICE).float()
    _, z_local = model.encode_local(patches)
    maps = z_local.reshape(len(indexes), grid, grid, model.n_local)[:, :, :, concept]
    return [upsample(maps[i]) for i in range(len(indexes))]


def show_images(indexes, title, heatmaps=None, scores=None):
    fig, axes = plt.subplots(1, len(indexes), figsize=(3 * len(indexes), 3.5))
    if len(indexes) == 1:
        axes = [axes]
    for number, (ax, index) in enumerate(zip(axes, indexes)):
        image, stem = load_image(index)
        ax.imshow(image.permute(1, 2, 0).clamp(0, 1).numpy())
        if heatmaps is not None:
            heatmap = heatmaps[number]
            vmin, vmax = float(heatmap.min()), float(heatmap.max())
            ax.imshow(heatmap, cmap="magma", alpha=0.5, vmin=vmin, vmax=max(vmax, vmin + 1e-8))
        caption = stem if scores is None else f"{stem}\n{scores[number]:.3f}"
        ax.set_title(caption, fontsize=8)
        ax.axis("off")
    fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    plt.show()


## Visualize Global concepts

Each row is one supercategory. The ten images are those on which its matched global concept activates most, and the number under each image is that code.


In [ ]:
for name, concept in zip(ontology.supercat_names, i_star):
    if concept < 0:
        print(f"{name}: no matched global concept")
        continue
    indexes = top_indexes(z_global[:, concept], TOP_K)
    scores = [float(z_global[index, concept]) for index in indexes]
    print(f"supercategory: {name}")
    show_images(indexes, f"global concept {concept}", scores=scores)


## Visualize Local concepts

Only parts of `EXAMPLE_SUPERCATEGORY`. Images are ranked by the mean of `j*` over patches. The heatmap overlay is that concept's code at each patch, resized to the CLIP crop.


In [ ]:
supercat_name = ontology.supercat_names[EXAMPLE_SUPERCATEGORY]
print(f"Example supercategory: {supercat_name}")
for part, concept in enumerate(j_star):
    if int(ontology.part_to_supercat[part]) != EXAMPLE_SUPERCATEGORY:
        continue
    name = ontology.part_names[part]
    if concept < 0:
        print(f"{name}: no matched local concept")
        continue
    indexes = top_indexes(z_local_pooled[:, concept], TOP_K)
    scores = [float(z_local_pooled[index, concept]) for index in indexes]
    print(f"Part: {name}")
    show_images(
        indexes,
        f"local concept {concept}",
        heatmaps=local_heatmaps(indexes, concept),
        scores=scores,
    )
